In [ ]:
!pip install -q whisper-timestamped
!pip install -q transformers

## Imports and config

In [ ]:
import torch
import torchaudio
import whisper_timestamped as whisper
import soundfile as sf
import difflib
import re
import gc
import json
from pathlib import Path
from tqdm import tqdm

TARGET_SR   = 16000
CHUNK_SEC   = 28          # max chunk duration (Whisper safe limit)
MAX_CHUNK_SEC = 28.0      # used when slicing aligned words into output chunks
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

TRAIN_AUDIO_DIR = Path('/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/audio')
TRAIN_TEXT_DIR  = TRAIN_AUDIO_DIR.parent / 'annotation'
OUTPUT_DIR      = Path('/kaggle/working/chunked_asr_train')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device      : {DEVICE}')
print(f'Audio dir   : {TRAIN_AUDIO_DIR}')
print(f'Output dir  : {OUTPUT_DIR}')

## Load models

In [ ]:
model = whisper.load_model(
    'bengaliAI/tugstugi_bengaliai-asr_whisper-medium',
    backend='transformers',
    device=DEVICE
)
model.model.generation_config.alignment_heads = [
    [5, 3], [5, 9], [8, 0], [8, 4], [8, 7], [8, 8],
    [9, 0], [9, 7], [9, 9], [10, 5]
]

# Force eager attention so output_attentions=True works for timestamp alignment
model.model.config._attn_implementation = 'eager'
print('Whisper model loaded with eager attention.')

In [ ]:
vad_model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    trust_repo=True,
    force_reload=False
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils
print('VAD model loaded.')

## Helper functions

In [ ]:
# ── Audio loading ──────────────────────────────────────────────────────────
def load_audio_mono(path, target_sr=TARGET_SR):
    audio, sr = torchaudio.load(path)
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    if sr != target_sr:
        audio = torchaudio.functional.resample(audio, sr, target_sr)
    return audio.squeeze(0)


# ── VAD: strip silence, return speech-only waveform ───────────────────────
def get_speech_waveform(waveform):
    speech_timestamps = get_speech_timestamps(waveform, vad_model, sampling_rate=TARGET_SR)
    if not speech_timestamps:
        return None, []
    parts = [waveform[ts['start']:ts['end']] for ts in speech_timestamps]
    return torch.cat(parts), speech_timestamps


# ── Transcribe speech waveform in 28s chunks ───────────────────────────────
def transcribe_in_chunks(waveform_numpy, sr=TARGET_SR, chunk_sec=CHUNK_SEC):
    chunk_samples = chunk_sec * sr
    total_samples = len(waveform_numpy)
    all_word_segments = []

    num_chunks = (total_samples + chunk_samples - 1) // chunk_samples

    for i in range(num_chunks):
        start  = i * chunk_samples
        end    = min(start + chunk_samples, total_samples)
        chunk  = waveform_numpy[start:end]
        offset = start / sr

        try:
            result = whisper.transcribe(
                model, chunk,
                language='bn',
                vad=False,
                naive_approach=False,
            )
            for segment in result.get('segments', []):
                for word in segment.get('words', []):
                    all_word_segments.append((
                        word['text'].strip(),
                        round(word['start'] + offset, 3),
                        round(word['end']   + offset, 3)
                    ))
        except Exception:
            pass  # skipped chunk — words will appear as None after alignment
        finally:
            del chunk
            if 'result' in locals():
                del result
            gc.collect()
            torch.cuda.empty_cache()

    return all_word_segments


# ── Clean Bengali fragment words ───────────────────────────────────────────
def is_valid_bengali_word(w):
    if len(w) < 2:
        return False
    invalid_starts = ('া', 'ে', 'ি', 'ো', 'ু', 'ূ', 'ৃ', 'ৈ', 'ৌ', 'ং', 'ঃ', 'ঁ')
    return not w.startswith(invalid_starts)


# ── Strip punctuation for matching only ───────────────────────────────────
def clean_word(w):
    # u0964 = Bengali danda, u0965 = double danda
    return re.sub(r'[\u0964\u0965,?!;:\'\u201c\u201d]', '', w).strip()


# ── Clean ground truth transcript ────────────────────────────────────────
# Compiled once at import time for efficiency
_DISALLOWED = re.compile(
    r'[^'
    r'\u0980-\u09FF'   # Bengali Unicode block (letters, vowel signs, danda, digits)
    r'\u200c\u200d'    # Zero-width non-joiner/joiner — required for Bengali conjuncts
    r'A-Za-z'           # English letters
    r'0-9'              # ASCII digits
    r' ,\.\?!\-\'"'  # Common punctuation + space
    r']'
)

def clean_transcript(text):
    """
    Keep only Bengali, English, digits and common punctuation.
    Remove all other scripts, control characters and corrupted text.
    """
    text = _DISALLOWED.sub(' ', text)  # replace disallowed with space
    text = re.sub(r' +', ' ', text)    # collapse multiple spaces
    return text.strip()


# ── Align Whisper words to ground truth ───────────────────────────────────
def align_to_ground_truth(word_segments, transcript):
    clean_segs = [(w, s, e) for w, s, e in word_segments if is_valid_bengali_word(w)]

    transcript        = clean_transcript(transcript)
    gt_words_original = transcript.split()
    gt_words_clean    = [clean_word(w) for w in gt_words_original]
    whisper_words     = [clean_word(w) for w, _, _ in clean_segs]

    matcher = difflib.SequenceMatcher(None, whisper_words, gt_words_clean, autojunk=False)
    aligned = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            for k in range(i2 - i1):
                w, start, end = clean_segs[i1 + k]
                aligned.append((gt_words_original[j1 + k], start, end))

        elif tag == 'replace':
            gt_span = gt_words_original[j1:j2]
            ws_span = clean_segs[i1:i2]
            for k in range(len(gt_span)):
                ws = ws_span[k] if k < len(ws_span) else None
                if ws:
                    aligned.append((gt_span[k], ws[1], ws[2]))
                else:
                    aligned.append((gt_span[k], None, None))

        elif tag == 'insert':
            for j in range(j1, j2):
                aligned.append((gt_words_original[j], None, None))
        # delete = hallucination, discard

    # Flaw 4 — match ratio sanity check
    ratio = matcher.ratio()
    if ratio < 0.5:
        print(f'  🔴🔴🔴🔴[warn] low alignment match ratio: {ratio:.3f} — result may be unreliable')

    # Flaw 3 — timestamp ordering validation
    prev_end = 0
    issues = 0
    for idx_w, (w, s, e) in enumerate(aligned):
        if s is not None:
            if s < prev_end - 1.0:  # allow 1s tolerance for rounding
                issues += 1
            prev_end = e if e is not None else prev_end
    if issues:
        print(f'  🔴🔴🔴🔴[warn] {issues} backwards timestamp(s) detected — alignment may have drift')

    return aligned


# ── Interpolate single missing timestamps, skip 2+ consecutive ────────────
def resolve_timestamps(aligned):
    """
    - Single None between two known timestamps: interpolate proportionally.
    - Two or more consecutive Nones: keep as None (will be skipped at chunk level).
    """
    result = list(aligned)
    n = len(result)
    i = 0
    while i < n:
        if result[i][1] is None:
            # find end of None block
            j = i
            while j < n and result[j][1] is None:
                j += 1
            block_size = j - i

            prev_end   = result[i - 1][2]   if i > 0 and result[i - 1][2]   is not None else None
            next_start = result[j][1]        if j < n and result[j][1]       is not None else None

            if block_size == 1 and prev_end is not None and next_start is not None:
                # single missing word — interpolate midpoint
                mid_s = round(prev_end, 3)
                mid_e = round(next_start, 3)
                result[i] = (result[i][0], mid_s, mid_e)
            # else: leave as None (2+ consecutive → will be dropped)

            i = j
        else:
            i += 1
    return result


# ── Slice aligned words into <=28s output chunks ───────────────────────────
def make_chunks(aligned_resolved, max_sec=MAX_CHUNK_SEC):
    """
    Walk through aligned words and produce chunks:
      - Normal words (with timestamps): accumulate up to max_sec, then flush.
      - None block (skipped transcription chunk):
          * Flush any in-progress normal chunk.
          * If the block has valid prev_end and next_start anchors AND fits
            within max_sec, save it as a gap chunk using those boundaries.
          * Otherwise skip entirely.
    Returns list of (words_list, start_sec, end_sec).
    """
    chunks = []
    cur_words = []
    cur_start = None
    cur_end   = None
    n = len(aligned_resolved)

    i = 0
    while i < n:
        word, ws, we = aligned_resolved[i]

        if ws is not None:
            # ── normal timestamped word ───────────────────────────────────
            if cur_start is None:
                cur_start = ws

            if we - cur_start > max_sec and cur_words:
                # flush current chunk before adding this word
                chunks.append((cur_words, cur_start, cur_end))
                cur_words = [word]
                cur_start = ws
                cur_end   = we
            else:
                cur_words.append(word)
                cur_end = we
            i += 1

        else:
            # ── start of a None block ─────────────────────────────────────
            block_start_idx = i
            while i < n and aligned_resolved[i][1] is None:
                i += 1
            block_end_idx = i  # index of first word after the gap

            # surrounding timestamp anchors
            prev_end = (
                aligned_resolved[block_start_idx - 1][2]
                if block_start_idx > 0
                and aligned_resolved[block_start_idx - 1][2] is not None
                else None
            )
            next_start = (
                aligned_resolved[block_end_idx][1]
                if block_end_idx < n
                and aligned_resolved[block_end_idx][1] is not None
                else None
            )

            gap_words = [
                aligned_resolved[j][0]
                for j in range(block_start_idx, block_end_idx)
            ]

            if prev_end is not None and next_start is not None:
                # flush any in-progress normal chunk first
                if cur_words:
                    chunks.append((cur_words, cur_start, cur_end))
                    cur_words, cur_start, cur_end = [], None, None

                # save gap as its own chunk if it fits within max_sec
                gap_dur = next_start - prev_end
                if 0 < gap_dur <= max_sec:
                    chunks.append((gap_words, prev_end, next_start))
                # else: zero-duration or too long — skip
            # else: no anchors on one or both sides — skip entirely

    # flush any remaining normal words
    if cur_words:
        chunks.append((cur_words, cur_start, cur_end))

    return chunks


# ── Save chunks to disk ────────────────────────────────────────────────────
def save_chunks(chunks, speech_waveform, base_name, output_dir, sr=TARGET_SR):
    file_dir   = output_dir / base_name
    audio_dir  = file_dir / 'chunk_audio'
    annot_dir  = file_dir / 'chunk_annotations'
    audio_dir.mkdir(parents=True, exist_ok=True)
    annot_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    for idx, (words, start_sec, end_sec) in enumerate(chunks, 1):
        start_sample = int(start_sec * sr)
        end_sample   = min(int(end_sec   * sr), len(speech_waveform))
        chunk_audio  = speech_waveform[start_sample:end_sample]

        if len(chunk_audio) == 0:
            continue

        wav_path = audio_dir  / f'{base_name}_chunk_{idx:04d}.wav'
        txt_path = annot_dir  / f'{base_name}_chunk_{idx:04d}.txt'

        sf.write(str(wav_path), chunk_audio.cpu().numpy(), sr)
        txt_path.write_text(' '.join(words), encoding='utf-8')
        saved += 1

    return saved


print('Helper functions defined.')

## Main pipeline — process all train audio files

In [ ]:
import time

audio_files = sorted(TRAIN_AUDIO_DIR.glob('*.wav'))
print(f'Found {len(audio_files)} audio files\n')

summary = []   # collect per-file stats for review

for audio_path in tqdm(audio_files, desc='Processing files'):
    base_name = audio_path.stem
    text_path = TRAIN_TEXT_DIR / f'{base_name}.txt'
    t0 = time.time()

    # ── skip if transcript missing ────────────────────────────────────────
    if not text_path.exists():
        print(f'🔴🔴🔴🔴🔴[warn] no transcript for {base_name}, skipping')
        summary.append({'file': base_name, 'status': 'no_transcript'})
        continue

    transcript = clean_transcript(text_path.read_text(encoding='utf-8').strip())
    if not transcript:
        print(f'🔴🔴🔴🔴🔴[warn] empty transcript for {base_name}, skipping')
        summary.append({'file': base_name, 'status': 'empty_transcript'})
        continue

    # ── load audio ────────────────────────────────────────────────────────
    try:
        waveform = load_audio_mono(audio_path)
    except Exception as e:
        print(f'🔴🔴🔴🔴🔴[warn] could not load {base_name}: {e}')
        summary.append({'file': base_name, 'status': f'load_error: {e}'})
        continue

    # ── VAD ───────────────────────────────────────────────────────────────
    try:
        speech_waveform, _ = get_speech_waveform(waveform)
        if speech_waveform is None or speech_waveform.numel() == 0:
            print(f'🔴🔴🔴🔴🔴[warn] no speech detected in {base_name}, skipping')
            summary.append({'file': base_name, 'status': 'no_speech'})
            del waveform
            gc.collect()
            continue
    except Exception as e:
        print(f'🔴🔴🔴🔴🔴[warn] VAD failed for {base_name}: {e}')
        summary.append({'file': base_name, 'status': f'vad_error: {e}'})
        del waveform
        gc.collect()
        continue
    finally:
        if 'waveform' in dir():
            del waveform
        gc.collect()

    # ── Transcribe ────────────────────────────────────────────────────────
    try:
        word_segments = transcribe_in_chunks(speech_waveform.cpu().numpy())
        if not word_segments:
            print(f'🔴🔴🔴🔴🔴[warn] transcription empty for {base_name}, skipping')
            summary.append({'file': base_name, 'status': 'transcription_empty'})
            del speech_waveform
            gc.collect()
            torch.cuda.empty_cache()
            continue
    except Exception as e:
        print(f'🔴🔴🔴🔴🔴[warn] transcription failed for {base_name}: {e}')
        summary.append({'file': base_name, 'status': f'transcription_error: {e}'})
        del speech_waveform
        gc.collect()
        torch.cuda.empty_cache()
        continue

    # ── Align ─────────────────────────────────────────────────────────────
    try:
        aligned          = align_to_ground_truth(word_segments, transcript)
        aligned_resolved = resolve_timestamps(aligned)

        if all(s is None for _, s, _ in aligned):
            print(f'🔴🔴🔴🔴🔴[warn] all timestamps None for {base_name}, skipping')
            summary.append({'file': base_name, 'status': 'all_timestamps_none'})
            del speech_waveform, word_segments, aligned
            gc.collect()
            torch.cuda.empty_cache()
            continue

    except Exception as e:
        print(f'🔴🔴🔴🔴🔴[warn] alignment failed for {base_name}: {e}')
        summary.append({'file': base_name, 'status': f'alignment_error: {e}'})
        del speech_waveform, word_segments
        gc.collect()
        torch.cuda.empty_cache()
        continue

    # ── Chunk and save ────────────────────────────────────────────────────
    try:
        chunks      = make_chunks(aligned_resolved)
        total_chunks = len(chunks)
        saved       = save_chunks(chunks, speech_waveform, base_name, OUTPUT_DIR)
        skipped     = total_chunks - saved
    except Exception as e:
        print(f'🔴🔴🔴🔴🔴[warn] saving failed for {base_name}: {e}')
        summary.append({'file': base_name, 'status': f'save_error: {e}'})
        del speech_waveform, word_segments, aligned, aligned_resolved
        gc.collect()
        torch.cuda.empty_cache()
        continue

    elapsed = time.time() - t0
    with_ts    = sum(1 for _, s, _ in aligned if s is not None)
    without_ts = sum(1 for _, s, _ in aligned if s is None)

    print(f'[ok] {base_name}: {saved}/{total_chunks} chunks saved '
          f'({skipped} skipped)'
          f'| aligned {with_ts}/{len(aligned)} words '
          f'| {elapsed:.0f}s')

    summary.append({
        'file'       : base_name,
        'status'     : 'ok',
        'chunks'     : saved,
        'words_total': len(aligned),
        'words_with_ts': with_ts,
        'words_no_ts': without_ts,
        'elapsed_s'  : round(elapsed, 1),
    })

    # ── Cleanup ───────────────────────────────────────────────────────────
    del speech_waveform, word_segments, aligned, aligned_resolved, chunks
    gc.collect()
    torch.cuda.empty_cache()

print('\n=== All files processed ===')

## Summary

In [ ]:
# Save summary JSON
summary_path = OUTPUT_DIR / 'processing_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'Summary saved to {summary_path}')

# Print summary table
ok      = [s for s in summary if s['status'] == 'ok']
errors  = [s for s in summary if s['status'] != 'ok']

print(f'\nSuccessfully processed : {len(ok)}/{len(summary)} files')
print(f'Errors / skipped       : {len(errors)}')

if ok:
    total_chunks = sum(s['chunks'] for s in ok)
    total_words  = sum(s['words_total'] for s in ok)
    total_with   = sum(s['words_with_ts'] for s in ok)
    print(f'Total chunks saved     : {total_chunks}')
    print(f'Total words            : {total_words}')
    print(f'Words with timestamps  : {total_with} ({100*total_with/total_words:.1f}%)')

if errors:
    print('\nFailed files:')
    for s in errors:
        print(f"  {s['file']}: {s['status']}")

## Package output as zip

In [ ]:
import shutil
zip_path = '/kaggle/working/chunked_asr_train.zip'
shutil.make_archive(
    base_name=zip_path.replace('.zip', ''),
    format='zip',
    root_dir='/kaggle/working',
    base_dir='chunked_asr_train'
)
print(f'Created {zip_path}')